# LIFE on Google Colab — PolitiFact++ (4-class, GPT-2 features)

Runs the LIFE pipeline end-to-end: **convert → key-sentence extraction → concatenate → features → train**.

**Before you start:**
1. Set the Colab runtime to **GPU** (Runtime → Change runtime type → T4 GPU).
2. Upload the whole `LIFE` repo (including `dataset/data/`) to your Google Drive, e.g. `MyDrive/LIFE`.
3. Edit `PROJECT_DIR` in the path cell below if you put it somewhere else.

Scope: **PolitiFact++** only (~520 articles). VLPFN is excluded (its text has no punctuation, so sentence splitting — which the whole method relies on — cannot work). GossipCop++ (~20k) is far heavier; try it only after this works.

In [ ]:
# Confirm a GPU is attached
!nvidia-smi

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# <-- change this if you uploaded the repo elsewhere
PROJECT_DIR = '/content/drive/MyDrive/LIFE'
os.chdir(PROJECT_DIR)

POLITIFACT_DIR = f'{PROJECT_DIR}/dataset/data/Fakenews-dataset-main/Fakenews-dataset-main/Dataset/PolitiFact++'
OUTPUT_RAW = f'{PROJECT_DIR}/dataset/output_raw'
KEY_SENT   = f'{PROJECT_DIR}/dataset/keySentence/important_sentences_top20.jsonl'
FEATURES   = f'{PROJECT_DIR}/dataset/features'
TRAIN_PATH = f'{PROJECT_DIR}/dataset/train.jsonl'
TEST_PATH  = f'{PROJECT_DIR}/dataset/test.jsonl'

print('cwd:', os.getcwd())
print('PolitiFact++ found:', os.path.isdir(POLITIFACT_DIR))

In [ ]:
# Install dependencies.
# If the fastNLP import fails at the training step, pin a compatible version:
#   !pip install -q fastNLP==1.0.1
!pip install -q -r requirements.txt

In [ ]:
# NLTK sentence tokenizer data. 'punkt' gives english.pickle (used by train.py);
# 'punkt_tab' is required by newer nltk's sent_tokenize (used by step 1).
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

## Step 0 — Convert PolitiFact++ JSON to JSONL
Produces `HF_fake.jsonl`, `MF_fake.jsonl`, `HR_true.jsonl`, `MR_true.jsonl` (97 / 97 / 194 / 132 lines).

In [ ]:
!python dataset/0_convert.py --input_dir "{POLITIFACT_DIR}" --output_dir "{OUTPUT_RAW}"

## Step 1 — Key-sentence extraction
Trains a BERT fake/real classifier (saved as `gpt3.5_bert_model.pt`), then finds the top-20 most impactful sentences per article. **This is the slowest step** (a forward pass per sentence per article); a few minutes to ~30 min on a T4 for PolitiFact++.

In [ ]:
!python dataset/1_keySentenceExtraction.py --data_dir "{OUTPUT_RAW}" --output_file "{KEY_SENT}" --gpu 0

## Step 2 — Concatenate key sentences back into the data
Adds a `sentence` field to each record in `OUTPUT_RAW` by matching on `(id, label)`. **Overwrites the files in `OUTPUT_RAW` in place** — re-run Step 0 first if you need to reset them.

In [ ]:
!python dataset/2_concate.py --folder_path "{OUTPUT_RAW}" --important_sentences_file "{KEY_SENT}"

## Step 3 — Generate GPT-2 fingerprint features (in-process)
Replaces the mosec server + HTTP client with direct GPT-2 log-likelihood scoring. Writes one feature JSONL per input file into `FEATURES`.

In [ ]:
!python dataset/3_gen_features_local.py --input_dir "{OUTPUT_RAW}" --output_dir "{FEATURES}" --model gpt2 --gpu 0

## Step 4 — Train the classifier
Splits `FEATURES` into train/test, then trains the Transformer+CRF sequence classifier over the 4 classes. Starts with **2 epochs as a smoke test** — raise `--num_train_epochs` (the repo default is 50) once it runs cleanly.

In [ ]:
!python LIFE_train/train.py \
  --split_dataset \
  --data_path "{FEATURES}" \
  --train_path "{TRAIN_PATH}" \
  --test_path "{TEST_PATH}" \
  --model Transformer \
  --num_train_epochs 2

## Notes / troubleshooting
- **fastNLP**: if step 4 errors on `from fastNLP.modules.torch import ...`, run `!pip install -q fastNLP==1.0.1` and restart the runtime.
- **Checkpoints**: `gpt3.5_bert_model.pt` (step 1) and `linear_en.pt` (step 4) are written to `PROJECT_DIR` on Drive, so they survive disconnects.
- **Scaling up**: point Step 0 at `.../Dataset/GossipCop++` and rerun — but expect Step 1 to take hours on a T4.
- **Re-runs**: Step 2 mutates `OUTPUT_RAW` in place; always re-run Step 0 before re-running Steps 1–3 from scratch.